# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
if not os.path.exists('/content/flyrank_internship'):
    !git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd /content/flyrank_internship

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)
low_ctr = (df['ctr'] < df['ctr'].median()).astype(int)
df['baseline_score'] = stale * visible * (1 + low_ctr) * df['impressions_90d']

baseline = df[['content_id','client_id','baseline_score','is_declining_label']].copy()
print("Loaded data:", len(df), "rows")
print("Rebuilt baseline in-memory:", len(baseline), "rows")

/content/flyrank_internship
Loaded data: 30000 rows
Rebuilt baseline in-memory: 30000 rows


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [3]:
print(df['is_declining_label'].value_counts(normalize=True))

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [4]:
numeric_features = ['word_count','char_count','ctr','avg_position','engagement_rate','scroll_rate',
    'ai_traffic_pct','content_age_days','days_since_last_update','search_volume','cpc',
    'impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d',
    'engaged_sessions_90d','ai_sessions_90d','scroll_events_90d',
    'days_with_impressions','days_with_sessions']
categorical_features = ['content_type','main_intent','competition_level','freshness_tier',
    'word_count_tier','char_count_tier','impression_tier','position_tier']

X = df[numeric_features].copy()
for col in numeric_features:
    X[f'has_{col}'] = X[col].notnull().astype(int)
    X[col] = X[col].fillna(0)
X_cat = pd.get_dummies(df[categorical_features], dummy_na=True)
X = pd.concat([X, X_cat], axis=1)
y = df['is_declining_label']
groups = df['client_id']

gkf = GroupKFold(n_splits=5)
print("Number of unique clients:", groups.nunique())
print("Feature matrix shape:", X.shape)

Number of unique clients: 32
Feature matrix shape: (30000, 81)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def cross_val_precision_at_k(model, X, y, groups, k, seed=42):
    scores_out = np.zeros(len(y))
    for train_idx, test_idx in gkf.split(X, y, groups):
        m = model.set_params(random_state=seed) if hasattr(model, 'random_state') else model
        m.fit(X.iloc[train_idx], y.iloc[train_idx])
        scores_out[test_idx] = m.predict_proba(X.iloc[test_idx])[:, 1]
    return precision_at_k(scores_out, y, k), scores_out

base_rate = y.mean()
baseline_p50 = precision_at_k(baseline['baseline_score'], baseline['is_declining_label'], 50)

logreg_p50, logreg_scores = cross_val_precision_at_k(LogisticRegression(max_iter=1000), X, y, groups, 50)
rf_p50, rf_scores = cross_val_precision_at_k(RandomForestClassifier(n_estimators=200, random_state=42), X, y, groups, 50)

results = pd.DataFrame({
    'method': ['base_rate', 'baseline_rule', 'logistic_regression', 'random_forest'],
    'precision_at_50': [base_rate, baseline_p50, logreg_p50, rf_p50]
})
print(results)


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

                method  precision_at_50
0            base_rate         0.542067
1        baseline_rule         0.680000
2  logistic_regression         0.600000
3        random_forest         0.580000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
final_model = RandomForestClassifier(n_estimators=200, random_state=42)
final_model.fit(X, y)

importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 feature importances:")
print(importances.head(10))

df['model_score'] = rf_scores
df['model_pred'] = (df['model_score'] > 0.5).astype(int)
wrong = df[df['model_pred'] != df['is_declining_label']]
print("\nTotal wrong predictions:", len(wrong))
print("\n3 concrete wrong cases:")
print(wrong[['content_id','avg_position','ctr','days_since_last_update','is_declining_label','model_score']].head(3))


Top 10 feature importances:
avg_position             0.091672
impressions_90d          0.089548
days_with_impressions    0.079476
content_age_days         0.072471
word_count               0.051992
char_count               0.051418
ctr                      0.043840
pageviews_90d            0.041943
days_with_sessions       0.038562
sessions_90d             0.038475
dtype: float64

Total wrong predictions: 11018

3 concrete wrong cases:
              content_id  avg_position    ctr  days_since_last_update  \
7   content_a63219c6e95a          21.2   0.06                      22   
13  content_a5a2fbc76336          39.8   0.00                     103   
15  content_689414059706           7.8  23.68                       8   

    is_declining_label  model_score  
7                    0        0.520  
13                   0        0.915  
15                   0        0.625  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.